In [0]:
# ============================================================
# BRONZE LAYER - CONFIGURATION
# ============================================================

from pyspark.sql.functions import (
    col,
    explode,
    to_json,
    current_timestamp,
    input_file_name
)

from datetime import date, timedelta


CATALOG = "fhir_assignment"
BRONZE_SCHEMA = "bronze"

RAW_BASE_PATH = "/Volumes/fhir_assignment/raw/fhir_files"


# ============================================================
# PIPELINE PARAMETER
# ============================================================

dbutils.widgets.text(
    "run_date",
    "",
    "Run Date"
)

run_date_param = dbutils.widgets.get("run_date").strip()


if run_date_param:
    run_date = date.fromisoformat(run_date_param)
else:
    run_date = date.today()


# 3-day inclusive window
start_date = run_date - timedelta(days=2)
end_date = run_date


INGESTION_DATE = (
    f"{start_date.isoformat()}_{end_date.isoformat()}"
)


FHIR_RESOURCES = [
    "Patient",
    "Encounter",
    "Observation",
    "Condition"
]

#parameters check
print(f"Run date        : {run_date}")
print(f"Raw data window : {INGESTION_DATE}")

from pyspark.sql.functions import col, explode, to_json, current_timestamp


def build_bronze_table(resource):

    raw_path = (
        f"{RAW_BASE_PATH}/"
        f"{resource}/"
        f"{INGESTION_DATE}/"
    )

    raw_df = (
        spark.read
        .option("multiLine", True)
        .option("pathGlobFilter", "page_[0-9]*.json")
        .json(raw_path)
    )

    bronze_df = (
        raw_df
        .select(
            explode(col("entry")).alias("entry"),
            col("_metadata.file_path").alias("source_file")
        )
        .select(
            col("entry.resource.resourceType").alias("resource_type"),
            col("entry.resource.id").alias("resource_id"),
            to_json(col("entry.resource")).alias("resource_json"),
            col("source_file"),
            current_timestamp().alias("ingestion_timestamp")
        )
    )

    return bronze_df

In [0]:
from pyspark.sql.functions import (
    col,
    explode,
    to_json,
    current_timestamp,
    sha2,
    concat_ws,
    lit
)

def build_bronze_table(resource):

    raw_path = (
        f"{RAW_BASE_PATH}/"
        f"{resource}/"
        f"{INGESTION_DATE}/"
    )

    raw_df = (
        spark.read
        .option("multiLine", True)
        .option("pathGlobFilter", "page_[0-9]*.json")
        .json(raw_path)
    )

    bronze_df = (
        raw_df
        .select(
            explode(col("entry")).alias("entry"),
            col("_metadata.file_path").alias("source_file")
        )
        .select(
            col("entry.resource.resourceType").alias("resource_type"),
            col("entry.resource.id").alias("resource_id"),
            to_json(col("entry.resource")).alias("resource_json"),
            col("source_file"),
            current_timestamp().alias("ingestion_timestamp")
        )
        .withColumn(
            "record_hash",
            sha2(col("resource_json"), 256)
        )
        .withColumn(
            "run_id",
            lit(INGESTION_DATE)
        )
    )

    # 3 day rolling window, want to keep scd type 2 for historical tracking but also remove duplicates thats why went with hash and runid to track data 

    return bronze_df

In [0]:
# ============================================================
# WRITE BRONZE TABLES - INCREMENTAL MERGE
# ============================================================

from delta.tables import DeltaTable

for resource in FHIR_RESOURCES:

    bronze_df = build_bronze_table(resource)

    table_name = (
        f"{CATALOG}.{BRONZE_SCHEMA}."
        f"{resource.lower()}"
    )

    target = DeltaTable.forName(
        spark,
        table_name
    )

    (
        target.alias("target")
        .merge(
            bronze_df.alias("source"),
            """
            target.resource_id = source.resource_id
            AND target.resource_type = source.resource_type
            """
        )
        .whenMatchedUpdate(
            condition="""
                target.record_hash <> source.record_hash
            """,
            set={
                "resource_json": "source.resource_json",
                "source_file": "source.source_file",
                "ingestion_timestamp": "source.ingestion_timestamp",
                "record_hash": "source.record_hash",
                "run_id": "source.run_id"
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )